# Day 1.3 — Structured Outputs

Instructions improved the answer but gave us nothing a program can rely on. Now we build a
real contract:

```text
Question -> model -> JSON that matches a schema -> Pydantic validation -> Python object
```

The schema tells the provider what shape to produce. Pydantic checks what actually arrived.
Both steps are needed, and they check different things.

## Before you begin

### Learning outcomes

- Define a Pydantic contract and read the JSON Schema it generates.
- Convert that schema into the **strict** form providers require, and explain each edit.
- Validate a response, and read a validation error instead of a crash.

Architecture reference: [D02](../../diagrams/source/day_01.md).

### Expected observation

Valid data becomes a typed Python object you can use with dot access. Data that is
plausible to a human but outside the field constraints is rejected, loudly and early.

## Concept briefing

## Why structured output matters

Free-form text is useful for people but unreliable for software. A program cannot safely
assume every response contains the same headings, fields or value types. A schema turns
this ambiguity into a contract. Validation does not make the model correct; it makes a
particular class of failure visible.

Consider a confidence field. The sentence "confidence is high" may be understandable to
a person but difficult to compare. A schema can require a number between 0 and 1. If the
model returns `4.5`, validation rejects it instead of quietly sending bad data deeper into
the application.

The correct mental model is:

- schema validity asks whether the response has an acceptable shape;
- factual evaluation asks whether its claims are correct;
- policy asks whether a requested action is permitted.

These are different checks and should not be collapsed into one model prompt.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — Write the contract as a Pydantic model

Each field states a name, a type, and where useful a constraint. This is the single place
where "what a research summary is" is defined for the rest of the application.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

class ResearchSummary(BaseModel):
    topic: str
    summary: str
    key_points: list[str] = Field(min_length=1, max_length=5)   # 1 to 5 bullet points
    confidence: float = Field(ge=0.0, le=1.0)                   # a probability, not a mood

print("Fields in the contract:")
for name, field in ResearchSummary.model_fields.items():
    print(f"  {name:12} {str(field.annotation):24} {field.metadata}")

### Step 2 — Look at the JSON Schema Pydantic generates

`model_json_schema()` turns the class into the machine-readable description we can hand to
a provider. Read it before sending it: two things in here will be rejected by strict mode.

In [ ]:
raw_schema = ResearchSummary.model_json_schema()
print(json.dumps(raw_schema, indent=2))
print()
print("Problem 1: there is no \"additionalProperties\": false, so extra invented fields are allowed.")
print("Problem 2: key_points carries minItems/maxItems, which strict mode does not support.")

### Step 3 — Convert it to the strict form

`"strict": True` promises the provider will **only** emit tokens that fit the schema. In
exchange the provider enforces a restricted schema dialect:

1. every object must declare `"additionalProperties": false`;
2. every object must list **all** of its properties in `required` (optional fields are
   expressed as a union with `null` instead);
3. the counting/measuring keywords (`minItems`, `maxItems`, `minLength`, `minimum`, …) are
   not part of that dialect and must be removed.

Sending a raw Pydantic schema with `strict: True` is the most common structured-output
bug: the provider returns a 400 and the notebook dies. Rule 3 does **not** weaken us —
Pydantic still enforces those limits locally, after the response arrives.

In [ ]:
# Keywords the strict dialect does not accept. Pydantic keeps enforcing them for us.
UNSUPPORTED = {
    "minItems", "maxItems", "minLength", "maxLength", "pattern", "format",
    "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "multipleOf", "default",
}

def make_strict(node):
    """Return a copy of a JSON Schema that satisfies the strict dialect.

    The function walks the whole tree, because nested objects (and objects inside
    arrays, and objects inside $defs) must follow the same three rules.
    """
    if isinstance(node, list):
        return [make_strict(item) for item in node]
    if not isinstance(node, dict):
        return node

    cleaned = {key: make_strict(value) for key, value in node.items() if key not in UNSUPPORTED}

    if cleaned.get("type") == "object":
        cleaned["additionalProperties"] = False          # rule 1
        cleaned["required"] = list(cleaned.get("properties", {}))   # rule 2
    return cleaned

strict_schema = make_strict(raw_schema)
print(json.dumps(strict_schema, indent=2))
print()
print("additionalProperties present:", strict_schema.get("additionalProperties"))
print("required lists every field  :", strict_schema.get("required"))
print("minItems removed            :", "minItems" not in json.dumps(strict_schema))

### Step 4 — Ask for the structured answer

The `response_format` block is where the schema goes. `provider.require_parameters`
tells OpenRouter to route only to providers that actually support structured output
instead of silently ignoring it.

In [ ]:
MOCK_STRUCTURED = json.dumps({
    "topic": "AI agents",
    "summary": "An application that uses a model to choose bounded actions.",
    "key_points": ["The host executes tools", "The loop needs a step limit"],
    "confidence": 0.9,
})

def ask_structured(prompt, max_tokens=600):
    """Return the raw response TEXT (not yet validated)."""
    if client is None:
        return MOCK_STRUCTURED
    try:
        response = client.chat.completions.create(
            model=COURSE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=max_tokens,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "research_summary",
                    "strict": True,
                    "schema": strict_schema,      # the CONVERTED schema, not raw_schema
                },
            },
            extra_body={
                "reasoning": {"effort": "low", "exclude": True},
                "provider": {"require_parameters": True},
            },
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("Live call failed, using the mock response ->", type(exc).__name__, exc)
        return MOCK_STRUCTURED

response_text = ask_structured("Explain an AI agent for a beginner with two or three key points.")
print("Raw text returned by the model:")
print(response_text)

### Step 5 — Validate, and explain failures in one line

`model_validate_json` either returns a typed object or raises. We wrap it once so a
failure prints something a beginner can act on — including the single most common live
failure, a truncated answer.

In [ ]:
def validate_or_explain(raw_text):
    """Return a ResearchSummary, or None after printing a readable diagnosis."""
    try:
        return ResearchSummary.model_validate_json(raw_text)
    except ValidationError as error:
        print("The response did NOT satisfy the contract:")
        print(error)
        if not raw_text.rstrip().endswith("}"):
            print()
            print("HINT: the text does not even end with '}' - it was cut off, not wrong.")
            print("      max_tokens counts the model's private reasoning tokens too, so a")
            print("      small max_tokens plus reasoning can end a response mid-JSON.")
            print("      Fix: raise max_tokens, or lower the reasoning effort.")
        return None

result = validate_or_explain(response_text)
print()
if result is not None:
    print("Validation passed. We now hold a typed object, not a string:")
    print("  type            :", type(result).__name__)
    print("  result.topic    :", result.topic)
    print("  result.confidence:", result.confidence)
    for number, point in enumerate(result.key_points, start=1):
        print(f"  key point {number}    : {point}")

### Step 6 — Break it: plausible data that is still invalid

Nothing about this JSON looks alarming to a person. `confidence: 4.5` would flow straight
into a dashboard, a threshold check, or a database column. Validation stops it here.

In [ ]:
invalid_data = """{
  "topic": "AI agents",
  "summary": "A short summary",
  "key_points": ["Uses a model"],
  "confidence": 4.5
}"""

print("--- confidence out of range ---")
validate_or_explain(invalid_data)

truncated = '{"topic": "AI agents", "summary": "An application that us'
print()
print("--- a truncated response ---")
validate_or_explain(truncated)

### Try it yourself

Add a fourth field `difficulty` that must be an integer from 1 to 5, then check that the
strict conversion still removes the range keywords while Pydantic still rejects a 9.

In [ ]:
# --- Worked solution ---
class EngineeringConcept(BaseModel):
    name: str
    explanation: str
    applications: list[str] = Field(min_length=1, max_length=3)
    difficulty: int = Field(ge=1, le=5)          # 1 = first year, 5 = research level

concept_schema = make_strict(EngineeringConcept.model_json_schema())
print("Strict schema keys for difficulty:", concept_schema["properties"]["difficulty"])
print("(ge/le became minimum/maximum in the raw schema and were then removed -")
print(" the provider does not enforce them, Pydantic does.)")
print()

# Shape is fine, value is not: exactly the case a schema alone would let through.
bad = '{"name": "Recursion", "explanation": "A function calling itself", ' \
      '"applications": ["Tree traversal"], "difficulty": 9}'
try:
    EngineeringConcept.model_validate_json(bad)
except ValidationError as error:
    print("Rejected by Pydantic, as intended:")
    print(error)

good = '{"name": "Recursion", "explanation": "A function calling itself", ' \
       '"applications": ["Tree traversal"], "difficulty": 2}'
print()
print("Accepted:", EngineeringConcept.model_validate_json(good))

### Checkpoint

**1. You send `strict: True` with the raw Pydantic schema and the provider answers with a 400 error. Name two edits that fix it.**

<details><summary>Show answer</summary>

Add `"additionalProperties": false` to every object and list every property in `required`;
and delete the keywords the strict dialect does not accept, such as `minItems`, `maxItems`
and `minLength`. That is exactly what `make_strict` does — and it must walk nested objects
too, not just the top level.

</details>

**2. Validation passed. Does that mean the summary is true?**

<details><summary>Show answer</summary>

No. Three different questions are easy to confuse:

- *shape* — does the response have the right fields and types? (schema + Pydantic)
- *truth* — are the claims correct? (evaluation, Day 2 onwards)
- *policy* — is this action allowed at all? (guardrails, Day 3)

A confidence of `0.9` is a valid float. It is not evidence of anything.

</details>

### Recap

- **Limitation we saw:** free-form text, and even a raw Pydantic schema, cannot be handed
  straight to a strict-mode provider or to `json.loads`.
- **Layer we added:** a schema converted to the strict dialect, plus Pydantic validation
  with a readable failure path including a truncation hint.
- **Evidence it worked:** the structured response validated into a typed object with dot
  access, while `confidence: 4.5` and a cut-off response were both rejected with an
  explanation.